In [1]:
"""
Figure S2 — GLORYS12 Validation: IGP2018, KH2025, and Argo Profile Comparisons
================================================================================
Six-panel figure (2 rows × 3 columns) showing GLORYS12 mean T/S profiles
against three independent observational datasets:

    Row 1 — Temperature (in-situ):
        a) Argo        — all available, Greenland Sea
        b) IGP2018     — CTD + XCTD, collocated in space/time, Feb–Mar 2018
        c) KH2025      — CTD, collocated in space/time, Feb–Mar 2025

    Row 2 — Salinity:
        d) Argo        — all available, Greenland Sea
        e) IGP2018     — CTD only (XCTD salinity also used where finite)
        f) KH2025      — CTD

All temperature panels use in-situ temperature for consistency:
  IGP2018 / KH2025 :  GLORYS thetao → in-situ T via TEOS-10 chain.
  Argo             :  T_argo stored as potential T in paired file;
                      converted back to in-situ T via gsw.t_from_CT.
  KH2025 obs       :  raw in-situ T from the .mat file.

Near-surface fix (configurable):
  KH2025 profiles often have contaminated / missing data in the upper
  NEAR_SURF_DEPTH metres.  Set NEAR_SURF_FIX to control replacement:
    'linear'      — linear extrapolation from the two shallowest clean levels
    'constant'    — fill with value at shallowest clean level
    'pchip'       — piecewise cubic extrapolation from clean data below
    'mixed_layer' — fill with mean of NEAR_SURF_DEPTH–MIXED_LAYER_BASE m
    None          — exclude data shallower than NEAR_SURF_DEPTH

Data shallower than DEPTH_MIN (10 m) excluded in all panels.

Output
------
    ./outputs/figures/figure_s02_GLORYS12_T_S_verification.png
    ./outputs/processed_data/figure_s02_mean_profiles.nc

Version:       2.1.0
Last Modified: 2026-06-23
Author:        Chris Barrell
"""

import warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import gsw
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.io import loadmat
from scipy.interpolate import PchipInterpolator

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Paths ---
IGP_BASE    = Path("/local/jbj13rpu/Documents/ROVER/IGP_data")
CTD_FILE    = IGP_BASE / "IGP_CTD.nc"
XCTD_FILE   = IGP_BASE / "IGP_XCTD.nc"
GLORYS_IGP  = IGP_BASE / "glorys12" / "glorys12_IGP_daily_FebMar2018.nc"

KH_MAT      = Path("/local/jbj13rpu/Documents/ROVER/KH_CTD/kh2025007002.mat")
GLORYS_KH   = Path("/local/jbj13rpu/Documents/ROVER/KH_CTD/glorys12/"
                   "glorys12_KH2025_daily_FebMar2025.nc")

ARGO_FILE   = Path("/local/jbj13rpu/Documents/ROVER/code/argo_glorys_paired_profiles.nc")

OUT_DIR     = Path("./outputs/figures")
OUT_FIG     = OUT_DIR / "figure_s02_GLORYS12_T_S_verification.png"
PROCESSED_DIR = Path("./outputs/processed_data/figure_s02")
PROCESSED_FILE = PROCESSED_DIR / "figure_s02_mean_profiles.nc"

# --- Repo flags ---
REPROCESS = False  # True: reprocess from raw data and save; False: load from cache

OUTPUT_METHODS_DIR = Path("./outputs/methods")
LOG_DIR            = Path("./outputs/logs")

# --- Near-surface fix for KH2025 ---
# One of: 'linear' | 'constant' | 'pchip' | 'mixed_layer' | 'clamped_cubic' | None
# None  → simply exclude depths < NEAR_SURF_DEPTH from KH2025 panels
NEAR_SURF_FIX    = None  # default: on
NEAR_SURF_DEPTH  = 30.0             # m — levels shallower than this are replaced / excluded
MIXED_LAYER_BASE = 60.0             # m — used only with 'mixed_layer' method
N_FIT_CLAMPED    = 20               # levels below NEAR_SURF_DEPTH used to anchor the cubic
                                    # at 1 dbar spacing: 20 levels = 30–50 m anchor zone

# --- Depth filter applied to all datasets ---
DEPTH_MIN = 10      # m — always exclude shallowest data (quality issues)
DEPTH_MAX = 1000    # m

# --- Plot colours ---
COL_OBS    = "black"
COL_GLORYS = "#2E86AB"

# =============================================================================
# MATLAB DATENUM HELPER
# =============================================================================

_MATLAB_EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)
_MATLAB_OFFSET = 719529  # datenum 719529 == 1970-01-01

def matlab_datenum_to_datetime(dn):
    return _MATLAB_EPOCH + timedelta(days=float(dn) - _MATLAB_OFFSET)

# =============================================================================
# TEOS-10 HELPERS
# =============================================================================
# GLORYS collocated files now contain precomputed t_insitu and SA/CT/sigma0
# (added by add_teos10_to_glorys_collocated.py).  No in-script conversion needed.

# =============================================================================
# NEAR-SURFACE FIX (for KH2025)
# =============================================================================

def fix_near_surface(press, temp, sal,
                     method=NEAR_SURF_FIX,
                     surf_depth=NEAR_SURF_DEPTH,
                     ml_base=MIXED_LAYER_BASE):
    """
    Replace / remove contaminated near-surface data (press <= surf_depth).

    Parameters
    ----------
    press, temp, sal : 1-D arrays (sorted shallow → deep)
    method : str or None
        'linear'      – linear extrapolation from the two shallowest clean levels
        'constant'    – fill with shallowest clean level value
        'pchip'       – PCHIP extrapolation from clean levels below surf_depth
        'mixed_layer' – fill with mean of surf_depth–ml_base range
        None          – mask values at press <= surf_depth with NaN
    Returns
    -------
    temp_out, sal_out : arrays of same shape as inputs
    """
    press = np.asarray(press, dtype=float)
    temp  = np.asarray(temp,  dtype=float).copy()
    sal   = np.asarray(sal,   dtype=float).copy()

    surf_mask  = press <= surf_depth          # levels to replace / remove
    clean_mask = press > surf_depth           # reliable data below surf_depth

    if surf_mask.sum() == 0 or clean_mask.sum() == 0:
        # Nothing to fix or no clean data to extrapolate from
        return temp, sal

    if method is None:
        temp[surf_mask] = np.nan
        sal[surf_mask]  = np.nan
        return temp, sal

    for var in [temp, sal]:
        clean_p = press[clean_mask]
        clean_v = var[clean_mask]
        valid_clean = np.isfinite(clean_v)
        if valid_clean.sum() < 2:
            var[surf_mask] = np.nan
            continue

        cp = clean_p[valid_clean]
        cv = clean_v[valid_clean]
        # Sort ascending by pressure (should already be, but be safe)
        order = np.argsort(cp)
        cp, cv = cp[order], cv[order]

        if method == 'constant':
            fill = cv[0]
            var[surf_mask] = fill

        elif method == 'linear':
            # Use two shallowest clean levels
            if len(cp) >= 2:
                slope = (cv[1] - cv[0]) / (cp[1] - cp[0])
                for idx in np.where(surf_mask)[0]:
                    var[idx] = cv[0] + slope * (press[idx] - cp[0])
            else:
                var[surf_mask] = cv[0]

        elif method == 'pchip':
            interp = PchipInterpolator(cp, cv, extrapolate=True)
            var[surf_mask] = interp(press[surf_mask])

        elif method == 'mixed_layer':
            ml_mask = (press >= surf_depth) & (press <= ml_base)
            if ml_mask.sum() >= 1:
                fill = np.nanmean(var[ml_mask])
            else:
                fill = cv[0]
            var[surf_mask] = fill

        elif method == 'clamped_cubic':
            # Fit a cubic through the shallowest N_FIT_CLAMPED clean levels
            # with the constraint that the gradient is zero at the surface
            # (press = 0), mimicking mixed-layer flattening.
            # Cubic f(p) = a*p^3 + b*p^2 + d  (no linear term => f'(0) = 0)
            cp_fit = cp[:N_FIT_CLAMPED]
            cv_fit = cv[:N_FIT_CLAMPED]
            A = np.column_stack([cp_fit**3, cp_fit**2, np.ones_like(cp_fit)])
            coeffs, _, _, _ = np.linalg.lstsq(A, cv_fit, rcond=None)
            a, b, d = coeffs
            for idx in np.where(surf_mask)[0]:
                p_i = press[idx]
                var[idx] = a * p_i**3 + b * p_i**2 + d

    return temp, sal


# =============================================================================
# IGP2018 DATA LOADING AND COLLOCATION
# =============================================================================

def load_igp2018_profiles(ctd_file, xctd_file):
    """Load IGP2018 CTD and XCTD profiles into a list of dicts."""
    profiles = []

    ds = xr.open_dataset(ctd_file)
    depths = ds["depth"].values
    for i in range(ds.sizes["profile"]):
        temp = ds["temperature"].values[i]
        sal  = ds["salinity"].values[i]
        if np.isfinite(temp).sum() < 5:
            continue
        profiles.append({
            "depth":  depths.copy(),
            "temp":   temp.copy(),
            "sal":    sal.copy(),
            "time":   pd.Timestamp(ds["time"].values[i]),
            "lat":    float(ds["lat"].values[i]),
            "lon":    float(ds["lon"].values[i]),
            "source": "CTD",
        })
    ds.close()

    ds = xr.open_dataset(xctd_file)
    depths = ds["depth"].values
    for i in range(ds.sizes["profile"]):
        temp = ds["temperature"].values[i]
        sal  = ds["salinity"].values[i] if "salinity" in ds else np.full_like(temp, np.nan)
        if np.isfinite(temp).sum() < 5:
            continue
        profiles.append({
            "depth":  depths.copy(),
            "temp":   temp.copy(),
            "sal":    sal.copy(),
            "time":   pd.Timestamp(ds["time"].values[i]),
            "lat":    float(ds["lat"].values[i]),
            "lon":    float(ds["lon"].values[i]),
            "source": "XCTD",
        })
    ds.close()

    print(f"  IGP2018 profiles loaded: {len(profiles)} "
          f"(CTD: {sum(p['source']=='CTD' for p in profiles)}, "
          f"XCTD: {sum(p['source']=='XCTD' for p in profiles)})")
    return profiles


# =============================================================================
# KH2025 DATA LOADING
# =============================================================================

def load_kh2025_profiles(mat_path, near_surf_fix=NEAR_SURF_FIX):
    """
    Load KH2025 CTD profiles from the .mat structured array.

    Structure: data[i] with fields lon, lat, time, station, section,
               press, temp, sal — all per-profile 1-D arrays.

    Applies near-surface contamination fix (configurable).
    Returns list of dicts with keys: depth, temp, sal, time, lat, lon, source
    """
    if not mat_path.exists():
        raise FileNotFoundError(f"KH2025 .mat not found: {mat_path}")

    print(f"  Loading KH2025: {mat_path}")
    raw  = loadmat(str(mat_path), squeeze_me=False)
    key  = [k for k in raw.keys() if not k.startswith('_')][0]
    data = raw[key].squeeze()
    n    = data.shape[0]

    profiles = []
    for i in range(n):
        try:
            lon  = float(data['lon'][i].squeeze())
            lat  = float(data['lat'][i].squeeze())
            dn   = float(data['time'][i].squeeze())
            ts   = pd.Timestamp(matlab_datenum_to_datetime(dn)).normalize()
            press = data['press'][i].squeeze().astype(float)
            temp  = data['temp'][i].squeeze().astype(float)
            sal   = data['sal'][i].squeeze().astype(float)

            # Align array lengths
            n_d = len(press)
            temp = temp[:n_d] if len(temp) >= n_d else np.concatenate(
                [temp, np.full(n_d - len(temp), np.nan)])
            sal  = sal[:n_d]  if len(sal)  >= n_d else np.concatenate(
                [sal,  np.full(n_d - len(sal),  np.nan)])

            # Near-surface fix (KH2025-specific)
            temp, sal = fix_near_surface(press, temp, sal, method=near_surf_fix)

            # Use pressure as depth approximation (dbar ≈ m)
            depth = press.copy()

            if np.isfinite(temp).sum() < 5:
                continue

            profiles.append({
                "depth":  depth,
                "temp":   temp,
                "sal":    sal,
                "time":   ts,
                "lat":    lat,
                "lon":    lon,
                "source": "CTD",
            })
        except Exception as e:
            print(f"    WARNING: KH2025 profile {i}: {e}")

    print(f"  KH2025 profiles loaded: {len(profiles)}")
    return profiles


# =============================================================================
# GLORYS COLLOCATION (shared by IGP2018 and KH2025)
# =============================================================================

def extract_glorys_collocated(profiles, glorys_file):
    """
    Collocate GLORYS12 profiles with each observational cast (bilinear, daily),
    then vertically interpolate onto the obs profile depth grid.

    Reads precomputed t_insitu and so directly from the GLORYS file
    (added by add_teos10_to_glorys_collocated.py) — no TEOS-10 conversion
    needed here.

    Returns list of dicts with keys: depth, temp (in-situ), sal.
    Depths below the local GLORYS bathymetry are masked NaN.
    None entries indicate no match.
    """
    ds_gl   = xr.open_dataset(glorys_file)
    gl_dep  = ds_gl["depth"].values
    if gl_dep[0] < 0:
        gl_dep = -gl_dep
    gl_times = pd.to_datetime(ds_gl["time"].values)
    lat_name = "latitude"  if "latitude"  in ds_gl.coords else "lat"
    lon_name = "longitude" if "longitude" in ds_gl.coords else "lon"

    if "t_insitu" not in ds_gl:
        raise KeyError(
            f"'t_insitu' not found in {glorys_file.name}. "
            "Run add_teos10_to_glorys_collocated.py first."
        )

    gl_profiles = []
    for p in profiles:
        t   = p["time"]
        idx = np.where(
            (gl_times.year  == t.year)  &
            (gl_times.month == t.month) &
            (gl_times.day   == t.day)
        )[0]
        if len(idx) == 0:
            gl_profiles.append(None)
            continue
        sl = ds_gl.isel(time=idx[0])
        try:
            prof = sl.interp({lat_name: float(p["lat"]), lon_name: float(p["lon"])},
                             method="linear")
        except Exception:
            try:
                prof = sl.sel({lat_name: float(p["lat"]), lon_name: float(p["lon"])},
                              method="nearest")
            except Exception:
                gl_profiles.append(None)
                continue

        # Read precomputed t_insitu and so from file
        t_native = prof["t_insitu"].values.astype(float)
        so_native = prof["so"].values.astype(float)
        valid_gl  = np.isfinite(t_native) & np.isfinite(so_native)

        if valid_gl.sum() < 2:
            gl_profiles.append(None)
            continue

        obs_dep  = np.asarray(p["depth"], dtype=float)
        gl_dep_v = gl_dep[valid_gl]
        t_v      = t_native[valid_gl]
        so_v     = so_native[valid_gl]
        seafloor = gl_dep_v.max()

        # Interpolate onto obs 1-m depth grid; outside GLORYS range → NaN
        t_interp = np.interp(obs_dep, gl_dep_v, t_v,  left=np.nan, right=np.nan)
        s_interp = np.interp(obs_dep, gl_dep_v, so_v, left=np.nan, right=np.nan)

        # Mask below local GLORYS seafloor
        t_interp[obs_dep > seafloor] = np.nan
        s_interp[obs_dep > seafloor] = np.nan

        gl_profiles.append({
            "depth": obs_dep,
            "temp":  t_interp,
            "sal":   s_interp,
        })

    ds_gl.close()
    matched = sum(g is not None for g in gl_profiles)
    print(f"  GLORYS matches: {matched}/{len(profiles)}")
    return gl_profiles


# =============================================================================
# ARGO PROFILE DICTS FROM PAIRED NETCDF
# =============================================================================

def load_argo_profile_dicts(argo_file):
    """
    Load Argo paired NetCDF and return four lists of profile dicts:
        argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl

    T_argo stored as potential temperature in the paired file; converted
    here to in-situ T via TEOS-10 so all temperature panels are consistent.
    """
    ds     = xr.open_dataset(argo_file)
    depths = ds["depth"].values
    n      = ds.dims["profile"]
    lats   = ds["lat"].values
    lons   = ds["lon"].values
    T_argo_raw   = ds["T_argo"].values
    T_glorys_raw = ds["T_glorys"].values
    S_argo_raw   = ds["S_argo"].values
    S_glorys_raw = ds["S_glorys"].values
    ds.close()

    argo_T_obs, argo_T_gl = [], []
    argo_S_obs, argo_S_gl = [], []

    for i in range(n):
        lat_i = float(lats[i])
        lon_i = float(lons[i])
        p_arr = gsw.p_from_z(-depths, lat_i)

        # Argo potential T → in-situ T
        pt_argo = T_argo_raw[i].astype(float)
        s_argo  = S_argo_raw[i].astype(float)
        t_argo_insitu = np.full_like(pt_argo, np.nan)
        valid = np.isfinite(pt_argo) & np.isfinite(s_argo) & np.isfinite(p_arr)
        if valid.sum() > 0:
            SA = gsw.SA_from_SP(s_argo[valid], p_arr[valid], lon_i, lat_i)
            CT = gsw.CT_from_pt(SA, pt_argo[valid])
            t_argo_insitu[valid] = gsw.t_from_CT(SA, CT, p_arr[valid])

        # GLORYS thetao → in-situ T
        pt_gl  = T_glorys_raw[i].astype(float)
        s_gl   = S_glorys_raw[i].astype(float)
        t_gl_insitu = np.full_like(pt_gl, np.nan)
        valid = np.isfinite(pt_gl) & np.isfinite(s_gl) & np.isfinite(p_arr)
        if valid.sum() > 0:
            SA = gsw.SA_from_SP(s_gl[valid], p_arr[valid], lon_i, lat_i)
            CT = gsw.CT_from_pt(SA, pt_gl[valid])
            t_gl_insitu[valid] = gsw.t_from_CT(SA, CT, p_arr[valid])

        argo_T_obs.append({"depth": depths, "T_argo":   t_argo_insitu})
        argo_T_gl.append( {"depth": depths, "T_glorys": t_gl_insitu})
        argo_S_obs.append({"depth": depths, "S_argo":   s_argo})
        argo_S_gl.append( {"depth": depths, "S_glorys": s_gl})

    print(f"  Argo profiles loaded and converted to in-situ T: {n}")
    return argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl, n



# =============================================================================
# PROCESSED DATA: SAVE AND LOAD
# =============================================================================

# Six dataset×variable combinations saved as mean binned profiles:
#   argo_T, argo_S, igp_T, igp_S, kh_T, kh_S
# Each has variables: depth, mean_obs, std_obs, n_obs,
#                      mean_gl,  std_gl,  n_gl

PANELS = [
    ("argo_T",  "T_argo",   "T_glorys"),
    ("argo_S",  "S_argo",   "S_glorys"),
    ("igp_T",   "temp",     "temp"),
    ("igp_S",   "sal",      "sal"),
    ("kh_T",    "temp",     "temp"),
    ("kh_S",    "sal",      "sal"),
]


def save_processed_data(argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl,
                         igp_obs_valid, igp_gl_valid,
                         kh_obs_valid,  kh_gl_valid):
    """
    Save the minimum processed data needed to replot Figure S2 without
    access to the original datasets.

    Saves mean binned profiles (depth, mean, std, n) for each of the six
    dataset×variable combinations to a single NetCDF file.
    """
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    obs_lists = {
        "argo_T": (argo_T_obs, "T_argo"),
        "argo_S": (argo_S_obs, "S_argo"),
        "igp_T":  (igp_obs_valid, "temp"),
        "igp_S":  (igp_obs_valid, "sal"),
        "kh_T":   (kh_obs_valid,  "temp"),
        "kh_S":   (kh_obs_valid,  "sal"),
    }
    gl_lists = {
        "argo_T": (argo_T_gl,  "T_glorys"),
        "argo_S": (argo_S_gl,  "S_glorys"),
        "igp_T":  (igp_gl_valid, "temp"),
        "igp_S":  (igp_gl_valid, "sal"),
        "kh_T":   (kh_gl_valid,  "temp"),
        "kh_S":   (kh_gl_valid,  "sal"),
    }

    data_vars = {}
    coords     = {}

    for key in obs_lists:
        obs_profs, var_obs = obs_lists[key]
        gl_profs,  var_gl  = gl_lists[key]

        od, om, os_, on = mean_profile(obs_profs, var=var_obs, depth_max=DEPTH_MAX)
        gd, gm, gs_, gn = mean_profile(gl_profs,  var=var_gl,  depth_max=DEPTH_MAX)

        # Use obs depth grid as reference (GLORYS interpolated onto it)
        depth_coord = f"{key}_depth"
        coords[depth_coord] = od

        data_vars[f"{key}_mean_obs"] = xr.Variable(depth_coord, om,
            attrs={"long_name": f"{key} obs mean profile",
                   "units": "degC" if "_T" in key else "psu"})
        data_vars[f"{key}_std_obs"]  = xr.Variable(depth_coord, os_,
            attrs={"long_name": f"{key} obs std profile",
                   "units": "degC" if "_T" in key else "psu"})
        data_vars[f"{key}_n_obs"]    = xr.Variable(depth_coord, on,
            attrs={"long_name": f"{key} obs sample count"})

        # Interpolate GLORYS mean onto obs depth grid for compact storage
        if len(gd) > 0 and len(od) > 0:
            gm_i  = np.interp(od, gd, gm,  left=np.nan, right=np.nan)
            gs_i  = np.interp(od, gd, gs_, left=np.nan, right=np.nan)
            gn_i  = np.interp(od, gd, gn.astype(float),
                              left=np.nan, right=np.nan)
        else:
            gm_i = gs_i = gn_i = np.full_like(od, np.nan)

        data_vars[f"{key}_mean_gl"]  = xr.Variable(depth_coord, gm_i,
            attrs={"long_name": f"{key} GLORYS12 mean profile",
                   "units": "degC" if "_T" in key else "psu"})
        data_vars[f"{key}_std_gl"]   = xr.Variable(depth_coord, gs_i,
            attrs={"long_name": f"{key} GLORYS12 std profile",
                   "units": "degC" if "_T" in key else "psu"})
        data_vars[f"{key}_n_gl"]     = xr.Variable(depth_coord, gn_i,
            attrs={"long_name": f"{key} GLORYS12 sample count"})

    ds_out = xr.Dataset(
        data_vars,
        coords=coords,
        attrs={
            "title":       "Figure S2 mean binned profiles — GLORYS12 validation",
            "description": (
                "Mean vertical profiles (1-m bins, DEPTH_MIN–DEPTH_MAX) for "
                "six dataset×variable combinations: Argo T/S, IGP2018 T/S, "
                "KH2025 T/S. obs = observations; gl = GLORYS12 collocated."
            ),
            "depth_min_m":  DEPTH_MIN,
            "depth_max_m":  DEPTH_MAX,
            "near_surf_fix": str(NEAR_SURF_FIX),
            "created":      pd.Timestamp.now().isoformat(),
        }
    )

    encoding = {v: {"zlib": True, "complevel": 4} for v in ds_out.data_vars}
    ds_out.to_netcdf(PROCESSED_FILE, encoding=encoding)
    print(f"  Processed data saved: {PROCESSED_FILE}")


def load_processed_data():
    """
    Load mean binned profiles from cache and reconstruct profile-dict lists
    compatible with draw_panel().

    Returns the same six profile-dict lists as the full pipeline:
        argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl,
        igp_obs_T_gl, igp_gl_T, igp_obs_S, igp_gl_S,
        kh_obs_T_gl,  kh_gl_T,  kh_obs_S,  kh_gl_S

    Each list contains a single dict representing the mean profile,
    which mean_profile() treats identically to a list of raw profiles.
    """
    ds = xr.open_dataset(PROCESSED_FILE)

    def _make_pair(key, var_obs, var_gl):
        depth = ds[f"{key}_depth"].values
        # Pack mean+std back into a single "profile" dict so mean_profile()
        # treats each depth bin as an individual observation with std=0
        # (the std is already encoded in the saved arrays and will be
        # reconstructed as-is by draw_panel via the returned lists).
        obs = [{"depth": depth, var_obs: ds[f"{key}_mean_obs"].values,
                f"_std": ds[f"{key}_std_obs"].values,
                f"_n":   ds[f"{key}_n_obs"].values}]
        gl  = [{"depth": depth, var_gl:  ds[f"{key}_mean_gl"].values,
                f"_std": ds[f"{key}_std_gl"].values,
                f"_n":   ds[f"{key}_n_gl"].values}]
        return obs, gl

    argo_T_obs, argo_T_gl = _make_pair("argo_T", "T_argo",   "T_glorys")
    argo_S_obs, argo_S_gl = _make_pair("argo_S", "S_argo",   "S_glorys")
    igp_obs_valid = [{"depth": ds["igp_T_depth"].values,
                      "temp":  ds["igp_T_mean_obs"].values,
                      "sal":   ds["igp_S_mean_obs"].values}]
    igp_gl_valid  = [{"depth": ds["igp_T_depth"].values,
                      "temp":  ds["igp_T_mean_gl"].values,
                      "sal":   ds["igp_S_mean_gl"].values}]
    kh_obs_valid  = [{"depth": ds["kh_T_depth"].values,
                      "temp":  ds["kh_T_mean_obs"].values,
                      "sal":   ds["kh_S_mean_obs"].values}]
    kh_gl_valid   = [{"depth": ds["kh_T_depth"].values,
                      "temp":  ds["kh_T_mean_gl"].values,
                      "sal":   ds["kh_S_mean_gl"].values}]
    ds.close()

    n_argo = int(ds["argo_T_n_obs"].values.sum())
    n_igp  = int(ds["igp_T_n_obs"].values.sum())
    n_ctd  = n_igp  # CTD count not separately stored; use total IGP
    n_kh   = int(ds["kh_T_n_obs"].values.sum())

    print(f"  Processed data loaded: {PROCESSED_FILE}")
    return (argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl,
            igp_obs_valid, igp_gl_valid,
            kh_obs_valid,  kh_gl_valid,
            n_argo, n_igp, n_ctd, n_kh)


# =============================================================================
# MEAN PROFILE ON 2-m BIN GRID
# =============================================================================

def mean_profile(profiles_list, var, depth_max=None):
    """
    Collect all (depth, value) pairs from a list of profile dicts,
    bin into 2-m intervals, return (bin_centres, mean, std, n).
    """
    all_d, all_v = [], []
    for p in profiles_list:
        if p is None:
            continue
        d = np.asarray(p["depth"], dtype=float)
        v = np.asarray(p[var],     dtype=float)
        ok = np.isfinite(v) & np.isfinite(d) & (d >= DEPTH_MIN)
        if depth_max is not None:
            ok &= (d <= depth_max)
        if ok.sum() == 0:
            continue
        all_d.extend(d[ok].tolist())
        all_v.extend(v[ok].tolist())

    if not all_d:
        return np.array([]), np.array([]), np.array([]), np.array([])

    all_d    = np.array(all_d)
    all_v    = np.array(all_v)
    bin_size = 1.0
    d_max    = depth_max if depth_max else all_d.max()
    edges    = np.arange(0, d_max + bin_size, bin_size)
    ctrs     = 0.5 * (edges[:-1] + edges[1:])
    means    = np.full(len(ctrs), np.nan)
    stds     = np.full(len(ctrs), np.nan)
    ns       = np.zeros(len(ctrs), dtype=int)

    for j in range(len(ctrs)):
        mask = (all_d >= edges[j]) & (all_d < edges[j + 1])
        if mask.sum() >= 2:
            means[j] = np.mean(all_v[mask])
            stds[j]  = np.std(all_v[mask])
            ns[j]    = mask.sum()

    ok = np.isfinite(means)
    return ctrs[ok], means[ok], stds[ok], ns[ok]


# =============================================================================
# STATS BOX
# =============================================================================

def bias_rmse_stats(obs_profiles, gl_profiles, var_obs, var_gl):
    """Compute depth-averaged bias (GLORYS − obs) and RMSE from mean profiles."""
    od, om, _, _ = mean_profile(obs_profiles, var=var_obs, depth_max=DEPTH_MAX)
    gd, gm, _, _ = mean_profile(gl_profiles,  var=var_gl,  depth_max=DEPTH_MAX)
    if len(od) == 0 or len(gd) == 0:
        return np.nan, np.nan
    shared = np.intersect1d(np.round(od).astype(int), np.round(gd).astype(int))
    if len(shared) < 5:
        return np.nan, np.nan
    om_i = np.interp(shared, od, om)
    gm_i = np.interp(shared, gd, gm)
    bias = float(np.mean(gm_i - om_i))
    rmse = float(np.sqrt(np.mean((gm_i - om_i) ** 2)))
    return bias, rmse


def add_stats_box(ax, bias, rmse, n_obs):
    """Add n / bias / RMSE annotation box to lower-right of axes."""
    txt = f"n = {n_obs}\nBias = {bias:+.3f}\nRMSE = {rmse:.3f}"
    ax.text(0.97, 0.03, txt, transform=ax.transAxes, fontsize=9,
            ha="right", va="bottom", family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="#aaaaaa", alpha=0.8),zorder=10)


# =============================================================================
# SINGLE PANEL DRAW
# =============================================================================

def draw_panel(ax, obs_profiles, gl_profiles, var_obs, var_gl,
               xlabel, obs_label, panel_label, n_obs, show_ylabel=False):
    """Draw one mean-profile panel with shading and stats box."""
    od, om, os_, _ = mean_profile(obs_profiles, var=var_obs, depth_max=DEPTH_MAX)
    gd, gm, gs_, _ = mean_profile(gl_profiles,  var=var_gl,  depth_max=DEPTH_MAX)

    if len(od) > 0:
        ax.fill_betweenx(od, om - os_, om + os_, color=COL_OBS,    alpha=0.12, zorder=2)
        ax.plot(om, od, color=COL_OBS,    lw=1.8, zorder=4, label=obs_label)
    if len(gd) > 0:
        ax.fill_betweenx(gd, gm - gs_, gm + gs_, color=COL_GLORYS, alpha=0.18, zorder=2)
        ax.plot(gm, gd, color=COL_GLORYS, lw=1.8, zorder=4, label="GLORYS12")

    ax.set_xlabel(xlabel, fontsize=11)
    ax.invert_yaxis()
    ax.set_ylim(bottom=DEPTH_MAX, top=0)
    ax.grid(True, linestyle="--", alpha=0.35, zorder=0)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(200))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(50))
    ax.legend(fontsize=10, loc="lower left", frameon=True, framealpha=0.9)
    ax.set_title(panel_label, fontsize=11, loc="left", fontweight="normal", pad=6)

    if show_ylabel:
        ax.set_ylabel("Depth (m)", fontsize=11)
    else:
        ax.set_ylabel("")

    bias, rmse = bias_rmse_stats(obs_profiles, gl_profiles, var_obs, var_gl)
    if np.isfinite(bias):
        add_stats_box(ax, bias, rmse, n_obs)


# =============================================================================
# MAIN
# =============================================================================

def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_METHODS_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    if REPROCESS or not PROCESSED_FILE.exists():
        # ---------------------------------------------------------- Argo
        print("Loading Argo paired profiles ...")
        argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl, n_argo = load_argo_profile_dicts(ARGO_FILE)

        # ------------------------------------------------------- IGP2018
        print("\nLoading IGP2018 observations ...")
        igp_obs = load_igp2018_profiles(CTD_FILE, XCTD_FILE)

        print("Collocating GLORYS12 to IGP2018 casts ...")
        igp_gl  = extract_glorys_collocated(igp_obs, GLORYS_IGP)

        igp_obs_valid = [p for p, g in zip(igp_obs, igp_gl) if g is not None]
        igp_gl_valid  = [g for g in igp_gl if g is not None]
        n_igp  = len(igp_obs_valid)
        n_ctd  = sum(p["source"] == "CTD" for p in igp_obs_valid)

        # ------------------------------------------------------- KH2025
        print(f"\nLoading KH2025 observations (near-surface fix: {NEAR_SURF_FIX}) ...")
        kh_obs = load_kh2025_profiles(KH_MAT, near_surf_fix=NEAR_SURF_FIX)

        print("Collocating GLORYS12 to KH2025 casts ...")
        kh_gl  = extract_glorys_collocated(kh_obs, GLORYS_KH)

        kh_obs_valid = [p for p, g in zip(kh_obs, kh_gl) if g is not None]
        kh_gl_valid  = [g for g in kh_gl if g is not None]
        n_kh   = len(kh_obs_valid)

        # ------------------------------------------------ Save to cache
        print("\nSaving processed data ...")
        save_processed_data(
            argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl,
            igp_obs_valid, igp_gl_valid,
            kh_obs_valid,  kh_gl_valid,
        )

    else:
        # -------------------------------------------- Load from cache
        print("Loading processed data from cache ...")
        (argo_T_obs, argo_T_gl, argo_S_obs, argo_S_gl,
         igp_obs_valid, igp_gl_valid,
         kh_obs_valid,  kh_gl_valid,
         n_argo, n_igp, n_ctd, n_kh) = load_processed_data()

    # ------------------------------------------------------------------ Plot
    print("\nPlotting ...")
    fig, axes = plt.subplots(2, 3, figsize=(12, 10), sharey=True)
    fig.subplots_adjust(wspace=0.06, hspace=0.28)

    # ---- Row 0 : Temperature ------------------------------------------- #
    draw_panel(
        axes[0, 0],
        argo_T_obs, argo_T_gl,
        var_obs="T_argo", var_gl="T_glorys",
        xlabel="In-situ temperature (°C)",
        obs_label="Argo",
        panel_label="a) Argo — temperature",
        n_obs=n_argo,
        show_ylabel=True,
    )

    draw_panel(
        axes[0, 1],
        igp_obs_valid, igp_gl_valid,
        var_obs="temp", var_gl="temp",
        xlabel="In-situ temperature (°C)",
        obs_label="IGP2018",
        panel_label="b) IGP2018 — temperature",
        n_obs=n_igp,
    )

    draw_panel(
        axes[0, 2],
        kh_obs_valid, kh_gl_valid,
        var_obs="temp", var_gl="temp",
        xlabel="In-situ temperature (°C)",
        obs_label="KH2025",
        panel_label="c) KH2025 — temperature",
        n_obs=n_kh,
    )

    # ---- Row 1 : Salinity ---------------------------------------------- #
    draw_panel(
        axes[1, 0],
        argo_S_obs, argo_S_gl,
        var_obs="S_argo", var_gl="S_glorys",
        xlabel="Practical salinity (psu)",
        obs_label="Argo",
        panel_label="d) Argo — salinity",
        n_obs=n_argo,
        show_ylabel=True,
    )

    draw_panel(
        axes[1, 1],
        igp_obs_valid, igp_gl_valid,
        var_obs="sal", var_gl="sal",
        xlabel="Practical salinity (psu)",
        obs_label="IGP2018",
        panel_label="e) IGP2018 — salinity",
        n_obs=n_ctd,
    )

    draw_panel(
        axes[1, 2],
        kh_obs_valid, kh_gl_valid,
        var_obs="sal", var_gl="sal",
        xlabel="Practical salinity (psu)",
        obs_label="KH2025",
        panel_label="f) KH2025 — salinity",
        n_obs=n_kh,
    )

    # near_surf_label = (
    #     f"near-surface fix: {NEAR_SURF_FIX} (top {NEAR_SURF_DEPTH:.0f} m)"
    #     if NEAR_SURF_FIX is not None
    #     else f"top {NEAR_SURF_DEPTH:.0f} m excluded"
    # )
    # fig.text(0.5, 0.01, f"KH2025 {near_surf_label}",
    #          ha="center", fontsize=9, color="#555555", style="italic")

    fig.savefig(OUT_FIG, dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"\nFigure saved: {OUT_FIG}")

main()


Loading Argo paired profiles ...
  Argo profiles loaded and converted to in-situ T: 3566

Loading IGP2018 observations ...
  IGP2018 profiles loaded: 304 (CTD: 190, XCTD: 114)
Collocating GLORYS12 to IGP2018 casts ...
  GLORYS matches: 304/304

Loading KH2025 observations (near-surface fix: None) ...
  Loading KH2025: /local/jbj13rpu/Documents/ROVER/KH_CTD/kh2025007002.mat
  KH2025 profiles loaded: 238
Collocating GLORYS12 to KH2025 casts ...
  GLORYS matches: 238/238

Saving processed data ...
  Processed data saved: outputs/processed_data/figure_s02/figure_s02_mean_profiles.nc

Plotting ...

Figure saved: outputs/figures/figure_s02_GLORYS12_T_S_verification.png
